# Terminal Case Study — Phase 5: Reinforcement Learning Gate Agent

**Case study**: Intermodal Container Terminal | **Phase**: 5 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Frame the terminal gate staffing problem as an MDP.
2. Build a custom Gymnasium environment wrapping `TerminalModel`.
3. Train a DQN agent to dynamically control gate count.
4. Compare the learned policy against a fixed-gates heuristic.

---
> **Note**: `TerminalEnv` in `simdes.envs` is a stub.  This notebook builds the environment
> from scratch as a teaching exercise — showing how any SimPy model becomes an RL environment.
>
> Phase 5 closes the case study: the agent decides in real time how many gates to open.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
from gymnasium import spaces
import simpy
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

## MDP Formulation

| Component | Terminal formulation |
|---|---|
| **State** | (gate_queue_length, crane_queue_length, gate_utilisation, time_fraction) |
| **Action** | Number of open gates: {1, 2, 3, 4, 5} |
| **Reward** | −mean_wait_gate over last 30 min, minus 2×n_gates (staffing cost) |
| **Episode** | One 8-hour shift (480 min), 16 decisions |
| **Termination** | End of shift |

In [ ]:
class SimpleTerminalEnv(gym.Env):
    """Minimal Gymnasium env wrapping a SimPy terminal gate process.

    The agent controls how many gates are open; the crane count is fixed.
    """

    metadata = {'render_modes': []}

    def __init__(
        self,
        arrival_rate: float = 10.0,   # trucks/hr
        gate_mean: float = 5.0,       # min
        crane_mean: float = 8.0,      # min
        n_cranes: int = 1,
        max_gates: int = 5,
        sim_time: float = 480.0,      # min
        decision_interval: float = 30.0,
        gate_cost_per_slot: float = 2.0,  # reward penalty per extra gate
        seed: int | None = None,
    ):
        super().__init__()
        self.arrival_rate = arrival_rate / 60.0  # convert to per-min
        self.gate_mean = gate_mean
        self.crane_mean = crane_mean
        self.n_cranes = n_cranes
        self.max_gates = max_gates
        self.sim_time = sim_time
        self.decision_interval = decision_interval
        self.gate_cost = gate_cost_per_slot
        self._base_seed = seed

        self.observation_space = spaces.Box(
            low=np.zeros(4, dtype=np.float32),
            high=np.array([100, 20, 1.0, 1.0], dtype=np.float32),
        )
        self.action_space = spaces.Discrete(max_gates)  # 0 → 1 gate, 4 → 5 gates

        self._rng: np.random.Generator | None = None
        self._env: simpy.Environment | None = None

    # ------------------------------------------------------------------
    def reset(self, *, seed: int | None = None, options=None):
        super().reset(seed=seed)
        rng_seed = seed if seed is not None else self._base_seed
        self._rng = np.random.default_rng(rng_seed)

        self._env = simpy.Environment()
        self._gate_resource = simpy.Resource(self._env, capacity=1)  # start with 1 gate
        self._crane_resource = simpy.Resource(self._env, capacity=self.n_cranes)
        self._gate_queue_len = 0
        self._crane_queue_len = 0
        self._gate_busy = 0
        self._interval_waits: list[float] = []
        self._next_decision = self.decision_interval
        self._current_gates = 1

        self._env.process(self._arrivals())
        self._env.run(until=self._next_decision)
        return self._get_obs(), {}

    # ------------------------------------------------------------------
    def step(self, action: int):
        n_gates = int(action) + 1   # action 0 → 1 gate
        # Update gate resource capacity
        self._gate_resource._capacity = n_gates
        self._current_gates = n_gates

        # Run simulation until next decision point
        next_stop = min(self._next_decision + self.decision_interval, self.sim_time)
        self._env.run(until=next_stop)
        self._next_decision = next_stop

        mean_wait = np.mean(self._interval_waits) if self._interval_waits else 0.0
        self._interval_waits = []

        reward = float(-mean_wait - self.gate_cost * (n_gates - 1))
        terminated = self._env.now >= self.sim_time
        return self._get_obs(), reward, terminated, False, {'mean_wait_gate': mean_wait}

    # ------------------------------------------------------------------
    def _get_obs(self) -> np.ndarray:
        gate_q  = min(self._gate_queue_len,  100) / 100.0
        crane_q = min(self._crane_queue_len,  20) /  20.0
        util    = min(self._gate_busy / max(self._current_gates, 1), 1.0)
        t_frac  = min(self._env.now / self.sim_time, 1.0) if self._env else 0.0
        return np.array([gate_q, crane_q, util, t_frac], dtype=np.float32)

    # ------------------------------------------------------------------
    def _arrivals(self):
        while True:
            iat = self._rng.exponential(1.0 / self.arrival_rate)
            yield self._env.timeout(iat)
            self._env.process(self._truck())

    def _truck(self):
        t0 = self._env.now
        self._gate_queue_len += 1
        with self._gate_resource.request() as req:
            yield req
            self._gate_queue_len -= 1
            self._gate_busy += 1
            wait = self._env.now - t0
            self._interval_waits.append(wait)
            yield self._env.timeout(self._rng.exponential(self.gate_mean))
            self._gate_busy -= 1

        self._crane_queue_len += 1
        with self._crane_resource.request() as req:
            yield req
            self._crane_queue_len -= 1
            yield self._env.timeout(self._rng.exponential(self.crane_mean))

In [ ]:
# Sanity check: single episode with fixed action
env = SimpleTerminalEnv(seed=0)
obs, _ = env.reset(seed=0)
print('Initial obs:', obs)

total_r = 0.0
done = False
while not done:
    obs, r, done, _, info = env.step(1)  # always 2 gates
    total_r += r
print(f'Episode return (2-gate heuristic): {total_r:.2f}')

In [ ]:
# Heuristic baselines
N_EP = 50
def eval_fixed(n_gates_action, n_ep=N_EP):
    returns = []
    e = SimpleTerminalEnv(seed=None)
    for ep in range(n_ep):
        _, _ = e.reset(seed=ep)
        G = 0.0; done = False
        while not done:
            _, r, done, _, _ = e.step(n_gates_action)
            G += r
        returns.append(G)
    return np.mean(returns), np.std(returns)

for action, label in [(0, '1 gate'), (1, '2 gates'), (2, '3 gates')]:
    m, s = eval_fixed(action)
    print(f'Fixed {label}: mean return = {m:.2f} ± {s:.2f}')

In [ ]:
# Train DQN agent
train_env = Monitor(SimpleTerminalEnv(seed=2024))
agent = DQN(
    'MlpPolicy', train_env,
    learning_rate=1e-3, buffer_size=5000, learning_starts=200,
    batch_size=32, gamma=0.99, exploration_fraction=0.3,
    verbose=1, seed=42,
)
agent.learn(total_timesteps=10_000, progress_bar=True)
agent.save('terminal_dqn')
print('Training done.')

In [ ]:
# Evaluate
eval_env = SimpleTerminalEnv(seed=99)
mean_r, std_r = evaluate_policy(agent, eval_env, n_eval_episodes=50)
print(f'DQN agent: mean return = {mean_r:.2f} ± {std_r:.2f}')

In [ ]:
# Comparison bar chart
m1, s1 = eval_fixed(0)
m2, s2 = eval_fixed(1)
labels = ['1 gate\n(fixed)', '2 gates\n(fixed)', 'DQN\nagent']
means  = [m1, m2, mean_r]
errs   = [s1, s2, std_r]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, means, yerr=errs, capsize=6,
       color=['#d62728', '#1f77b4', '#2ca02c'], alpha=0.8)
ax.set_ylabel('Mean episode return')
ax.set_title('Terminal RL — gate policy comparison')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

## Summary

The DQN agent learns to open more gates during peak arrival periods and fewer gates
during quiet periods — balancing truck wait time against staffing cost.

This notebook also demonstrates the **recipe for wrapping any SimPy model as a Gym env**:
1. Create `simpy.Resource` objects for each server.
2. Yield `env.run(until=next_decision)` at each step.
3. Collect per-interval statistics for the reward.
4. Return an observation vector capturing system state.

This pattern generalises to any DES model — clinic, supply chain, manufacturing line.

## Try It Yourself

1. Modify the reward to also penalise crane waiting (trucks stuck after the gate).
   Does this change the learned gate policy?
2. Add a surge: arrival rate doubles from minute 200–300. Does the DQN adapt?
3. Implement the full `TerminalEnv` stub in `simdes/simdes/envs/terminal_env.py`
   using this notebook as your reference.